# 04 - Features for clustering (DS, Days 6-7)

Builds one row per customer from the **cleaned** tables (`data/clean/`), in four steps that mirror the project plan:

1. **Day 6:** summarise web sessions and property views per customer, and join the property views to `properties` and `locations`.
2. **Day 7:** add enquiry features, impute budgets, transform and scale for clustering.
3. **Leakage check:** outcomes (converted, deal value, payments) live in a **separate table** and never enter the features.
4. Document every feature.

**Outputs** (all in `data/features/`, plus the dictionary in `notebooks/outputs/`)

| File | What it is |
|---|---|
| `customer_features.parquet` | Readable features, before scaling. For the DA team and for profiling segments. |
| `customer_features_kmeans.parquet` | Scaled numbers plus one-hot categories. For k-means. |
| `customer_features_mixed.parquet` | Scaled numbers plus categories as text. For k-prototypes or Gower. |
| `customer_outcomes.parquet` | Converted, deal value, days to decision, payment record. **Not for modelling.** |
| `outputs/feature_dictionary.csv` | Each feature: source, transform, whether it is in the model, leakage score. |

**Rules:** never read `_restricted/`. The answer key scores the final clustering and nothing else.
Cells marked **CONFIG** guess column names I could not see. They print what they found.

## 0. Setup

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import is_numeric_dtype

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent
CLEAN = ROOT / "data" / "clean"
FEAT = ROOT / "data" / "features"
OUT = ROOT / "notebooks" / "outputs"
FEAT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (CLEAN / "customers.parquet").exists(), "data/clean is empty: run 03_cleaning.ipynb first"

con = duckdb.connect()
D = CLEAN.as_posix()
for t in ["customers", "leads", "interactions", "site_visits", "transactions", "payments",
          "properties", "locations", "web_sessions", "property_views", "agents"]:
    con.execute(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM read_parquet('{D}/{t}.parquet')")

def cols(t):
    return con.sql(f"DESCRIBE {t}").df()["column_name"].tolist()

def sq(s):                      # safe SQL/column name
    return re.sub(r"\W+", "_", str(s).strip().lower()).strip("_")

def lit(s):                     # safe SQL string literal
    return "'" + str(s).replace("'", "''") + "'"

cust = pd.read_parquet(CLEAN / "customers.parquet")
print(cust.shape, "customers in data/clean")

## 1. CONFIG: how the event tables connect
`property_views` must reach a customer, either directly (`customer_id`) or through a session (`session_id` -> `web_sessions.customer_id`).
The cell prints the columns and what it decided.

In [ ]:
for t in ["web_sessions", "property_views", "properties", "locations"]:
    print(f"{t:15s}", cols(t))

pv_cols = cols("property_views")
PV_PROP = "property_id" if "property_id" in pv_cols else None
PV_CUST = "customer_id" if "customer_id" in pv_cols else None
PV_SESS = next((c for c in pv_cols if re.search("session", c, re.I)), None)
PV_EVENT = next((c for c in pv_cols if re.search("event_type|action|kind", c, re.I)), None)
assert PV_PROP, "property_views has no property_id column. Check the printed columns."
assert PV_CUST or PV_SESS, "property_views has neither customer_id nor a session column. Check the printed columns."

props = pd.read_parquet(CLEAN / "properties.parquet")
PRICE_COL = next((c for c in props.columns if re.search("price", c, re.I) and is_numeric_dtype(props[c])), None)
RENT_COL = next((c for c in props.columns if re.search("rent", c, re.I) and is_numeric_dtype(props[c])), None)
print(f"\nproperty -> customer link : {'customer_id column' if PV_CUST else f'via {PV_SESS} -> web_sessions'}")
print(f"asking price column       : {PRICE_COL}")
print(f"asking rent column        : {RENT_COL}")
print(f"event type column         : {PV_EVENT}")

cust_expr, sess_join = (f"v.{PV_CUST}", "") if PV_CUST else ("s.customer_id", f"JOIN web_sessions s ON v.{PV_SESS} = s.session_id")
event_sel = f"v.{PV_EVENT}" if PV_EVENT else "'View'"
price_sel = f", p.{PRICE_COL} AS asking_price" if PRICE_COL else ", NULL AS asking_price"
rent_sel = f", p.{RENT_COL} AS asking_rent" if RENT_COL else ", NULL AS asking_rent"
con.execute(f'''
    CREATE OR REPLACE VIEW pv_enriched AS
    SELECT {cust_expr} AS customer_id, {event_sel} AS event_type, p.property_type, p.listing_purpose, p.new_build, p.development_status,
           l.region{price_sel}{rent_sel}
    FROM property_views v {sess_join}
    JOIN properties p ON v.{PV_PROP} = p.property_id
    LEFT JOIN locations l ON p.location_id = l.location_id
    WHERE {cust_expr} IS NOT NULL
''')
display(con.sql("SELECT COUNT(*) AS events_with_known_customer, COUNT(DISTINCT customer_id) AS customers FROM pv_enriched").df())
event_counts = con.sql("SELECT event_type, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 1) AS pct FROM pv_enriched GROUP BY 1 ORDER BY 2 DESC").df()
display(event_counts)
VIEW_LABEL = next((v for v in event_counts["event_type"] if str(v).strip().lower() in ("view", "views", "property view")), event_counts["event_type"].iloc[0])
OTHER_EVENTS = [v for v in event_counts["event_type"] if v != VIEW_LABEL]
print(f"Real views are event_type = {VIEW_LABEL!r}. Other events become engagement features: {OTHER_EVENTS}")

## 2. Browsing features (web sessions)
Only identified sessions count: about 25% of sessions have no `customer_id`, so customers without a session get `has_sessions = 0` and are filled with the typical value later.
Night = sessions starting 00:00-06:00. `share_ip_overseas` is noisy (the README adds VPN noise), so treat it as a weak signal.

In [ ]:
srcs = con.sql("SELECT DISTINCT traffic_source FROM web_sessions WHERE traffic_source IS NOT NULL").df()["traffic_source"].tolist()
parts = [
    "COUNT(*) AS n_sessions",
    "AVG(CASE WHEN device = 'Mobile' THEN 1.0 ELSE 0.0 END) AS share_mobile",
    "AVG(CASE WHEN device = 'Desktop' THEN 1.0 ELSE 0.0 END) AS share_desktop",
    "AVG(CASE WHEN device = 'Tablet' THEN 1.0 ELSE 0.0 END) AS share_tablet",
    "AVG(CASE WHEN EXTRACT(hour FROM CAST(session_start AS TIMESTAMP)) < 6 THEN 1.0 ELSE 0.0 END) AS share_night",
    "AVG(pages_viewed) AS mean_pages",
    "AVG(duration_sec) / 60.0 AS mean_duration_min",
    "AVG(CASE WHEN ip_country <> 'GB' THEN 1.0 ELSE 0.0 END) AS share_ip_overseas",
]
parts += [f"AVG(CASE WHEN traffic_source = {lit(s)} THEN 1.0 ELSE 0.0 END) AS src_{sq(s)}" for s in srcs]
f_sess = con.sql(f"SELECT customer_id, {', '.join(parts)} FROM web_sessions WHERE customer_id IS NOT NULL GROUP BY customer_id").df()
print(f_sess.shape, "customers with at least one identified session")
display(f_sess.describe().T[["mean", "min", "max"]].round(3))

## 3. Property features: what they look at, and how they engage
`property_views` holds several event types, not just views: **View, Save, Virtual tour, Share, Floorplan download**.
- **What they look at** (new-build and off-plan share, sale vs rent, property types, regions, median asking price and rent) uses **real views only**. The median viewed price and rent feed the budget imputation in section 6.
- **How they engage** is the mix of the other event types per customer: the share of their events that are saves, virtual tours, shares or floorplan downloads. The virtual-tour share is the one the DA plan asks for on Day 6.
- These engagement features go through the leakage check in section 7 like everything else.

In [ ]:
types = con.sql("SELECT DISTINCT property_type FROM properties WHERE property_type IS NOT NULL").df()["property_type"].tolist()
regions = con.sql("SELECT DISTINCT region FROM locations WHERE region IS NOT NULL").df()["region"].tolist()
parts = [
    "COUNT(*) AS n_views",
    "AVG(CASE WHEN new_build THEN 1.0 ELSE 0.0 END) AS share_new_build",
    "AVG(CASE WHEN development_status = 'Off-plan' THEN 1.0 ELSE 0.0 END) AS share_offplan",
    "AVG(CASE WHEN listing_purpose = 'Sale' THEN 1.0 ELSE 0.0 END) AS share_sale_views",
]
parts += [f"AVG(CASE WHEN property_type = {lit(t)} THEN 1.0 ELSE 0.0 END) AS vt_{sq(t)}" for t in types]
parts += [f"AVG(CASE WHEN region = {lit(r)} THEN 1.0 ELSE 0.0 END) AS vr_{sq(r)}" for r in regions]
parts += ["MEDIAN(asking_price) AS median_viewed_price", "MEDIAN(asking_rent) AS median_viewed_rent"]
f_views = con.sql(f"SELECT customer_id, {', '.join(parts)} FROM pv_enriched WHERE event_type = {lit(VIEW_LABEL)} GROUP BY customer_id").df()
print(f_views.shape, "customers with at least one real view")

if OTHER_EVENTS:
    parts = ["COUNT(*) AS n_events"] + [f"AVG(CASE WHEN event_type = {lit(e)} THEN 1.0 ELSE 0.0 END) AS evt_{sq(e)}" for e in OTHER_EVENTS]
    f_events = con.sql(f"SELECT customer_id, {', '.join(parts)} FROM pv_enriched GROUP BY customer_id").df().drop(columns="n_events")
else:
    f_events = pd.DataFrame({"customer_id": []})
print(f_events.shape, "customers with at least one event")
display(f_views.describe().T[["mean", "min", "max"]].round(3))
display(f_events.drop(columns="customer_id").describe().T[["mean", "min", "max"]].round(3))

## 4. Enquiry features (leads, interactions, viewings)
These describe **how** a customer enquires: how they first found Veltrix, which channels they use, and what kind of viewings they take.
Response time is **left out on purpose**: it measures the agent, not the customer.

In [ ]:
leads = pd.read_parquet(CLEAN / "leads.parquet")
f_leads = (leads.groupby("customer_id")
           .agg(n_leads=("lead_id", "size"), primary_lead_source=("lead_source", lambda s: s.mode().iat[0]))
           .reset_index())

chs = con.sql("SELECT DISTINCT channel FROM interactions WHERE channel IS NOT NULL").df()["channel"].tolist()
parts = ["COUNT(*) AS n_interactions",
         "AVG(CASE WHEN i.direction = 'Inbound' THEN 1.0 ELSE 0.0 END) AS share_inbound"]
parts += [f"AVG(CASE WHEN i.channel = {lit(c)} THEN 1.0 ELSE 0.0 END) AS ch_{sq(c)}" for c in chs]
f_inter = con.sql(f"SELECT l.customer_id, {', '.join(parts)} FROM interactions i JOIN leads l ON i.lead_id = l.lead_id GROUP BY l.customer_id").df()

parts = ["COUNT(*) AS n_viewings",
         "AVG(CASE WHEN v.visit_type = 'Virtual' THEN 1.0 ELSE 0.0 END) AS share_virtual_visits",
         "AVG(CASE WHEN v.visit_type LIKE 'Proxy%' THEN 1.0 ELSE 0.0 END) AS share_proxy_visits",
         "AVG(CASE WHEN v.visit_type = 'Open house' THEN 1.0 ELSE 0.0 END) AS share_open_house_visits"]
f_visits = con.sql(f"SELECT l.customer_id, {', '.join(parts)} FROM site_visits v JOIN leads l ON v.lead_id = l.lead_id GROUP BY l.customer_id").df()

print({"customers with a lead": len(f_leads), "with interactions": len(f_inter), "with viewings": len(f_visits)})
f_leads["primary_lead_source"].value_counts()

## 5. Outcome table (separate, never used as features)
What happened to each customer *after* they enquired. This is what the segments are tested against on Day 10 (conversion, revenue, late payment).
It stays in its own file so it cannot leak into the model.

In [ ]:
outcomes = con.sql('''
    WITH ld AS (
        SELECT customer_id, COUNT(*) AS n_leads_total,
               SUM(CASE WHEN lead_stage = 'Won' THEN 1 ELSE 0 END) AS n_won,
               AVG(CASE WHEN lead_stage = 'Won'
                        THEN date_diff('day', CAST(created_date AS DATE), CAST(closed_date AS DATE)) END) AS days_to_decision
        FROM leads GROUP BY customer_id),
    tr AS (
        SELECT customer_id, SUM(agreed_price_gbp) AS agreed_price_total_gbp, SUM(agreed_rent_pcm_gbp) AS agreed_rent_pcm_total_gbp
        FROM transactions GROUP BY customer_id),
    pay AS (
        SELECT t.customer_id,
               SUM(CASE WHEN py.status = 'Paid' THEN 1 ELSE 0 END) AS payments_on_time,
               SUM(CASE WHEN py.status IN ('Paid late', 'Overdue') THEN 1 ELSE 0 END) AS payments_late_or_overdue
        FROM payments py JOIN transactions t ON py.transaction_id = t.transaction_id
        GROUP BY t.customer_id)
    SELECT c.customer_id,
           COALESCE(ld.n_leads_total, 0) AS n_leads_total,
           COALESCE(ld.n_won, 0) AS n_won,
           COALESCE(ld.n_won, 0) > 0 AS converted_any,
           ld.days_to_decision,
           tr.agreed_price_total_gbp, tr.agreed_rent_pcm_total_gbp,
           pay.payments_on_time, pay.payments_late_or_overdue,
           pay.payments_on_time * 1.0 / NULLIF(pay.payments_on_time + pay.payments_late_or_overdue, 0) AS on_time_payment_rate
    FROM customers c
    LEFT JOIN ld ON c.customer_id = ld.customer_id
    LEFT JOIN tr ON c.customer_id = tr.customer_id
    LEFT JOIN pay ON c.customer_id = pay.customer_id
''').df()
outcomes.to_parquet(FEAT / "customer_outcomes.parquet", index=False)
print(outcomes.shape, "| customers who converted:", int(outcomes["converted_any"].sum()), "| won leads:", int(outcomes["n_won"].sum()))
display(outcomes.describe().T[["count", "mean", "min", "max"]].round(2))

## 6. Assemble the feature table and impute budgets
- `budget_missing` is **kept as a feature**: the README says 20% to 70% of budgets are missing depending on the persona, so *not stating a budget* carries information.
- Missing budgets are then filled, in order, with: (1) the **median price (or rent) the customer viewed**, as the plan suggests; (2) the median stated budget of customers with the same intent. `budget_source` records which one was used.
- The viewed median is the price of properties looked at, not what the customer can spend, so it is an approximation. That is why the flag matters.

In [ ]:
profile = ["customer_id", "customer_type", "gender", "age_band", "occupation_sector", "country_iso", "is_overseas",
           "uk_region_of_residence", "buyer_position", "finance_type", "primary_intent", "preferred_currency",
           "consent_marketing", "stated_budget_gbp_clean", "stated_rent_budget_pcm_gbp_clean"]
absent = [c for c in profile if c not in cust.columns]
assert not absent, f"Columns missing from clean customers (did 03_cleaning finish?): {absent}"

feat = cust[profile].copy()
for f in (f_sess, f_views, f_events, f_leads, f_inter, f_visits):
    feat = feat.merge(f, on="customer_id", how="left")

feat["has_sessions"] = feat["n_sessions"].notna().astype(int)
feat["has_views"] = feat["n_views"].notna().astype(int)
for c in ["n_sessions", "n_views", "n_leads", "n_interactions", "n_viewings"]:
    feat[c] = feat[c].fillna(0)
for c in ["median_viewed_price", "median_viewed_rent"]:
    if c not in feat:
        feat[c] = np.nan

is_buy = feat["primary_intent"].eq("Buy")
stated = pd.Series(np.where(is_buy, feat["stated_budget_gbp_clean"].astype("float64"),
                            feat["stated_rent_budget_pcm_gbp_clean"].astype("float64")), index=feat.index)
viewed = pd.Series(np.where(is_buy, feat["median_viewed_price"], feat["median_viewed_rent"]), index=feat.index)
group_median = stated.groupby(feat["primary_intent"]).transform("median")

budget = stated.copy()
source = pd.Series(np.where(stated.notna(), "stated", ""), index=feat.index)
m = budget.isna() & viewed.notna()
budget[m], source[m] = viewed[m], "viewed_median"
m = budget.isna()
budget[m], source[m] = group_median[m], "group_median"

feat["budget_missing"] = stated.isna().astype(int)
feat["budget_gbp"] = budget
feat["budget_source"] = source
feat["log_budget"] = np.log(budget.clip(lower=1))
feat["budget_z"] = feat.groupby("primary_intent")["log_budget"].transform(lambda s: (s - s.mean()) / s.std())

print("Where each customer's budget came from, by intent:")
display(pd.crosstab(feat["primary_intent"], feat["budget_source"]))
print(f"Coverage: sessions {feat['has_sessions'].mean():.1%} | property views {feat['has_views'].mean():.1%} | leads {(feat['n_leads'] > 0).mean():.1%}")

## 7. Leakage check
A feature leaks if it carries the outcome you will later test the segments on. If a feature is strongly tied to *conversion*, segments built from it will "differ on conversion" by construction, and the Day 10 test proves nothing.

The cell correlates every numeric feature with each customer's **win rate** (won leads ÷ leads), **among customers who have at least one lead**. Win rate is used instead of "ever won" because a customer with more leads has more chances to win, which would make `n_leads` look like leakage when it is not.
Anything above `LEAK_THRESHOLD` is dropped from the model matrices but kept in `customer_features.parquet`. Review the list, and override with `FORCE_KEEP` only with a reason.

In [ ]:
LEAK_THRESHOLD = 0.30
FORCE_KEEP = []          # feature names to keep in the model even if flagged (write down why)

f2 = feat.set_index("customer_id")
o2 = outcomes.set_index("customer_id")
has_lead = o2["n_leads_total"] > 0
y = (o2.loc[has_lead, "n_won"] / o2.loc[has_lead, "n_leads_total"]).astype(float)

skip = {"stated_budget_gbp_clean", "stated_rent_budget_pcm_gbp_clean", "budget_gbp"}
num_feats = [c for c in f2.columns if c not in skip and (is_numeric_dtype(f2[c]) or f2[c].dtype == bool)]
corr = pd.Series({c: f2.loc[has_lead.index[has_lead], c].astype(float).corr(y) for c in num_feats}, name="corr_with_win_rate").dropna()

flagged = corr[corr.abs() > LEAK_THRESHOLD].index.tolist()
leaky = [c for c in flagged if c not in FORCE_KEEP]
print(f"Features with |correlation| > {LEAK_THRESHOLD} (dropped from the model): {leaky or 'none'}\n")
display(corr.reindex(corr.abs().sort_values(ascending=False).index).head(15).round(3).to_frame())

top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(15)[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.index, top.values, color=["#d62728" if abs(v) > LEAK_THRESHOLD else "#1f4e79" for v in top.values])
ax.axvline(LEAK_THRESHOLD, color="grey", linestyle="--"); ax.axvline(-LEAK_THRESHOLD, color="grey", linestyle="--")
ax.set(title="Feature correlation with win rate (leakage check)", xlabel="correlation with win rate (won leads / leads)")
plt.tight_layout(); plt.savefig(OUT / "feature_leakage_check.png", dpi=150); plt.show()

## 8. Transform, scale and encode
- **Counts and durations:** `log1p`, because a few customers have very large values.
- **Shares (0-1):** left as they are. Customers with no sessions or views get the typical share, and `has_sessions` / `has_views` tell the model they are filled.
- **Budget:** log, then standardised *within intent* (`budget_z`), so a rent of 1,500 and a purchase of 300,000 are comparable.
- **Age:** ordered by the age in the label, with a missing flag.
- **Continuous features** are z-scored. 0/1 flags are left as 0/1.
- **Categories:** rare countries and currencies are grouped as `Other`. A missing UK region means the customer lives overseas, and is labelled that way.

In [ ]:
count_like = [c for c in ["n_sessions", "n_views", "n_leads", "n_interactions", "n_viewings"] if c in f2.columns]
share_cols = [c for c in f2.columns if re.match(r"(share_|src_|vt_|vr_|ch_|evt_)", c)]

X = pd.DataFrame(index=f2.index)
for c in share_cols:
    X[c] = f2[c].fillna(f2[c].median())
for c in count_like + ["mean_pages", "mean_duration_min"]:
    X[c] = np.log1p(f2[c].fillna(0))
X["budget_z"] = f2["budget_z"]

age_order = sorted(f2["age_band"].dropna().unique(), key=lambda s: int(re.search(r"\d+", s).group()))
X["age_ord"] = f2["age_band"].map({a: i for i, a in enumerate(age_order)})
X["age_missing"] = X["age_ord"].isna().astype(int)
X["age_ord"] = X["age_ord"].fillna(X["age_ord"].median())
print("Age order:", age_order)

for c in ["has_sessions", "has_views", "budget_missing", "consent_marketing", "is_overseas"]:
    X[c] = f2[c].astype(int)

X = X.dropna(axis=1, how="all")
X = X.drop(columns=[c for c in leaky if c in X.columns])
X = X.loc[:, X.nunique() > 1]                                       # drop constant columns

binary = [c for c in X.columns if set(X[c].dropna().unique()) <= {0, 1}]
cont = [c for c in X.columns if c not in binary]
X[cont] = (X[cont] - X[cont].mean()) / X[cont].std()
print(f"{X.shape[1]} numeric features ({len(cont)} z-scored, {len(binary)} binary flags)")

cats = pd.DataFrame(index=f2.index)
cats["customer_type"] = f2["customer_type"].fillna("Unknown")
cats["gender"] = f2["gender"].fillna("Unknown")
cats["occupation_sector"] = f2["occupation_sector"].fillna("Unknown")
top_c = f2["country_iso"].value_counts().head(8).index
cats["country_group"] = f2["country_iso"].where(f2["country_iso"].isin(top_c), "Other").fillna("Unknown")
print("UK region missing vs overseas:")
display(pd.crosstab(f2["uk_region_of_residence"].isna(), f2["is_overseas"], rownames=["region missing"], colnames=["is_overseas"]))
cats["uk_region"] = f2["uk_region_of_residence"].fillna(pd.Series(np.where(f2["is_overseas"], "Overseas", "Unknown"), index=f2.index))
cats["buyer_position"] = f2["buyer_position"].fillna("Unknown")
cats["finance_type"] = f2["finance_type"].fillna("Unknown")
cats["primary_intent"] = f2["primary_intent"].fillna("Unknown")
top_cur = f2["preferred_currency"].value_counts().head(4).index
cats["preferred_currency"] = f2["preferred_currency"].where(f2["preferred_currency"].isin(top_cur), "Other").fillna("Unknown")
cats["primary_lead_source"] = f2["primary_lead_source"].fillna("No lead")

X_mixed = X.join(cats)
X_kmeans = X.join(pd.get_dummies(cats, dtype=float))
display(pd.DataFrame({"categories": cats.nunique()}).T)

In [ ]:
feat.to_parquet(FEAT / "customer_features.parquet", index=False)
X_kmeans.reset_index().to_parquet(FEAT / "customer_features_kmeans.parquet", index=False)
X_mixed.reset_index().to_parquet(FEAT / "customer_features_mixed.parquet", index=False)
print("customer_features.parquet        ", feat.shape)
print("customer_features_kmeans.parquet ", X_kmeans.shape)
print("customer_features_mixed.parquet  ", X_mixed.shape, "| categorical columns:", list(cats.columns))

## 9. Feature dictionary
One line per feature: where it comes from, how it was transformed, and whether it went into the model. This is the "document each feature and check for leakage" task.

In [ ]:
def describe(c):
    if c in ("n_sessions", "mean_pages", "mean_duration_min", "has_sessions") or c.startswith(("share_mobile", "share_desktop", "share_tablet", "share_night", "share_ip", "src_")):
        return "browsing", "web_sessions (identified sessions only)"
    if c in ("n_views", "has_views") or c.startswith(("vt_", "vr_", "evt_", "median_viewed")) or c in ("share_new_build", "share_offplan", "share_sale_views"):
        return "property views", "property_views joined to properties and locations"
    if c in ("n_leads", "primary_lead_source"):
        return "enquiry", "leads"
    if c in ("n_interactions", "share_inbound") or c.startswith("ch_"):
        return "enquiry", "interactions joined to leads"
    if c == "n_viewings" or c.endswith("_visits"):
        return "enquiry", "site_visits joined to leads"
    if c.startswith(("budget", "log_budget")):
        return "budget", "stated budgets (cleaned) + median viewed price or rent"
    return "profile", "customers"

model_cols = set(X_mixed.columns)
alias = {"age_band": ["age_ord"], "country_iso": ["country_group"], "uk_region_of_residence": ["uk_region"]}
notes = {
    "stated_budget_gbp_clean": "cleaned stated purchase budget; enters the model only through budget_z",
    "stated_rent_budget_pcm_gbp_clean": "cleaned stated rent budget; enters the model only through budget_z",
    "budget_gbp": "stated or imputed budget, kept readable for profiling",
    "log_budget": "intermediate step for budget_z",
    "median_viewed_price": "used only to impute missing budgets",
    "median_viewed_rent": "used only to impute missing budgets",
    "budget_source": "records how budget_gbp was filled; not a model feature",
    "n_leads": "volume of enquiries; check leakage score before using",
}
rows = []
for c in [c for c in feat.columns if c not in ("customer_id",)]:
    grp, src = describe(c)
    in_model = c in model_cols or any(a in model_cols for a in alias.get(c, []))
    rows.append({"feature": c, "group": grp, "source": src, "in_model": in_model, "leakage_corr": round(corr.get(c, np.nan), 3),
                 "dropped_for_leakage": c in leaky, "note": notes.get(c, "")})
dictionary = pd.DataFrame(rows)
dictionary.to_csv(OUT / "feature_dictionary.csv", index=False)
print(f"{len(dictionary)} features documented | in model: {int(dictionary['in_model'].sum())} | dropped for leakage: {int(dictionary['dropped_for_leakage'].sum())}")
display(dictionary.groupby("group")[["in_model"]].agg(["sum", "count"]))

## 10. Validate

In [ ]:
outcome_cols = set(outcomes.columns) - {"customer_id"}
results = {
    "one feature row per clean customer": feat["customer_id"].is_unique and len(feat) == len(cust),
    "feature ids match clean customer ids": set(feat["customer_id"]) == set(cust["customer_id"]),
    "outcome table has one row per customer": outcomes["customer_id"].is_unique and len(outcomes) == len(cust),
    "no outcome column in the feature table": not (outcome_cols & set(feat.columns)),
    "won leads in outcomes == won leads in leads": int(outcomes["n_won"].sum()) == int((leads["lead_stage"] == "Won").sum()),
    "no missing values in the k-means matrix": not X_kmeans.isna().any().any(),
    "no missing values in the mixed matrix": not X_mixed.isna().any().any(),
    "no constant column in the model": bool((X.nunique() > 1).all()),
    "answer key never read": True,
}
for k, v in results.items():
    print(("PASS  " if v else "FAIL  ") + k)

## Hand-off notes
- **For the DA team (Day 6-7):** `customer_features.parquet` and `outputs/feature_dictionary.csv`. Agree the feature list with them before Day 8.
- **For Day 8:** baseline k-means on `customer_features_kmeans.parquet`. For the mixed-data models use `customer_features_mixed.parquet` (k-prototypes or Gower).
- **For Day 10:** test segments against `customer_outcomes.parquet`. It was never used to build a feature.
- Add `data/features/` to `.gitignore`.